In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/data/train_cleaned.csv')
test = pd.read_csv('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/data/test_cleaned.csv')

# CREATE INTERACTION FEATURE
train['price_x_store_age'] = train['product_price'] * train['store_age_years']
test['price_x_store_age'] = test['product_price'] * test['store_age_years']

# Prepare data
X_train = train.drop(['id', 'product_code', 'store_code', 'total_sales'], axis=1)
y_train = train['total_sales']
X_test = test.drop(['id', 'product_code', 'store_code'], axis=1)

# Load preprocessor
with open('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/models/preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

# Transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Data loaded and preprocessed ✓")
print(f"Training data shape: {X_train_processed.shape}")
print(f"Target shape: {y_train.shape}")

Data loaded and preprocessed ✓
Training data shape: (6818, 28)
Target shape: (6818,)


In [2]:
# Train validation split
from sklearn.model_selection import train_test_split

# Split: 80% train, 20% validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_processed, y_train, test_size=0.2, random_state=42
)

print(f"Training set: {X_train_split.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

Training set: 5454 samples
Validation set: 1364 samples


In [3]:
import optuna
from optuna.samplers import TPESampler
import xgboost as xgb
from sklearn.model_selection import cross_val_score
import numpy as np

print("\n" + "="*70)
print("PHASE 6: HYPERPARAMETER TUNING - XGBoost (OPTUNA)")
print("="*70)

# Define objective function
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1)
    }
    
    # Create model with trial parameters
    model = xgb.XGBRegressor(**params, random_state=42, verbosity=0)
    
    # Cross-validation score
    scores = cross_val_score(
        model, X_train_split, y_train_split,
        cv=5, scoring='neg_mean_squared_error', n_jobs=-1
    )
    
    # Return RMSE (negative to positive)
    rmse = np.sqrt(-scores.mean())
    return rmse

# Create study with TPE sampler (Bayesian optimization)
sampler = TPESampler(seed=42)
study = optuna.create_study(
    sampler=sampler,
    direction='minimize'  # Minimize RMSE
)

print("\n⏳ Optuna tuning in progress (10-20 trials)...\n")

# Optimize
study.optimize(
    objective,
    n_trials=20,  # Number of trials (faster than GridSearchCV)
    show_progress_bar=True,
    n_jobs=1  # Optuna handles parallelization internally
)

print("\n" + "="*70)
print("OPTUNA TUNING COMPLETE")
print("="*70)

# Best trial
best_trial = study.best_trial
print(f"\n✓ Best RMSE: {best_trial.value:.4f}")
print(f"\n✓ Best Parameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

[I 2026-09-25 05:51:47,788] A new study created in memory with name: no-name-baecf9c5-24ee-42be-b755-ace81f5da605



PHASE 6: HYPERPARAMETER TUNING - XGBoost (OPTUNA)

⏳ Optuna tuning in progress (10-20 trials)...



  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-09-25 05:52:15,689] Trial 0 finished with value: 1189.9998616768994 and parameters: {'learning_rate': 0.11861663446573512, 'max_depth': 10, 'n_estimators': 233, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'gamma': 0.2904180608409973, 'reg_alpha': 0.8661761457749352, 'reg_lambda': 0.6011150117432088}. Best is trial 0 with value: 1189.9998616768994.
[I 2026-09-25 05:52:19,643] Trial 1 finished with value: 1137.2445616762639 and parameters: {'learning_rate': 0.21534104756085318, 'max_depth': 3, 'n_estimators': 293, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'min_child_weight': 2, 'gamma': 0.9170225492671691, 'reg_alpha': 0.3042422429595377, 'reg_lambda': 0.5247564316322378}. Best is trial 1 with value: 1137.2445616762639.
[I 2026-09-25 05:52:22,612] Trial 2 finished with value: 1163.7053136694722 and parameters: {'learning_rate': 0.13526405540621358, 'max_depth': 5, 'n_estimators': 203, 'subsample': 

In [4]:
# Create model with best parameters
best_xgb_optuna = xgb.XGBRegressor(
    **best_trial.params,
    random_state=42,
    verbosity=0
)

# Train on full training split
best_xgb_optuna.fit(X_train_split, y_train_split)

# Evaluate
y_pred_optuna = best_xgb_optuna.predict(X_val)
rmse_optuna = np.sqrt(mean_squared_error(y_val, y_pred_optuna))
r2_optuna = r2_score(y_val, y_pred_optuna)

print("\n" + "="*70)
print("OPTUNA TUNED XGBoost - VALIDATION PERFORMANCE")
print("="*70)
print(f"Validation RMSE: {rmse_optuna:.4f}")
print(f"Validation R²: {r2_optuna:.4f}")


OPTUNA TUNED XGBoost - VALIDATION PERFORMANCE
Validation RMSE: 1101.4148
Validation R²: 0.5880


In [5]:
print("\n" + "="*70)
print("PHASE 7: FINAL PREDICTIONS & SUBMISSION")
print("="*70)

# Use tuned model to predict on test set
y_test_pred = best_xgb_optuna.predict(X_test_processed)

# Create submission DataFrame
submission = pd.DataFrame({
    'id': test['id'],
    'total_sales': y_test_pred
})

# Save to CSV
submission.to_csv('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/submission/final_submission_2.csv', index=False)

print("\n✓ Submission file created: submissions/final_submission.csv")
print(f"\nSubmission Preview:")
print(submission.head(10))
print(f"\nTotal predictions: {len(submission)}")
print(f"Sales range: {submission['total_sales'].min():.2f} - {submission['total_sales'].max():.2f}")


PHASE 7: FINAL PREDICTIONS & SUBMISSION

✓ Submission file created: submissions/final_submission.csv

Submission Preview:
          id  total_sales
0  row_00009  2487.293457
1  row_00015  4768.232910
2  row_00019  3406.643555
3  row_00020  3054.440674
4  row_00023  2464.551025
5  row_00026   454.320953
6  row_00027  3488.259521
7  row_00033  3145.816162
8  row_00034   274.716827
9  row_00037  1942.913208

Total predictions: 1705
Sales range: 64.39 - 7221.54
